In [26]:
import os
import json 
from labelme import utils
from pycocotools import mask as maskUtils
import numpy as np
from pathlib import Path

In [27]:
def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)

    x_min = xs.min()
    x_max = xs.max()
    y_min = ys.min()
    y_max = ys.max()

    return [
        int(x_min),
        int(y_min),
        int(x_max - x_min + 1),
        int(y_max - y_min + 1)
    ]

In [34]:
current_directory = os.getcwd()
print(current_directory)

def create_bulk_coco_annotations_from_polygon_images(labelme_annotation_dir, output_coco_json_file):
    output_coco_json_data = {
        "info": {},
        "licenses": [],
        "categories": [
            {
                "supercategory": "foreground",
                "id": 1,
                "name": "1-Layer"
            },
            {
                "supercategory": "foreground",
                "id": 2,
                "name": "2-Layer"
            },
            {
                "supercategory": "foreground",
                "id": 3,
                "name": "3-Layer"
            },
            {
                "supercategory": "foreground",
                "id": 4,
                "name": "4-Layer"
            }
        ]
    }

    coco_images = []
    images_with_idx = dict()
    cur_image_idx = 0
    cur_polygon_idx = 0
    annotations = []

    input_json_dir = Path(labelme_annotation_dir)

    for json_path in input_json_dir.glob("*.json"):
        with open(json_path) as f:
            data = json.load(f)

            image_full_path = os.path.abspath(os.path.join(os.path.dirname(json_path), data["imagePath"]))
            image_file_name = os.path.basename(image_full_path)
            print(image_file_name, image_full_path)
            if image_file_name not in images_with_idx:
                images_with_idx[image_file_name] = cur_image_idx
                coco_images.append({
                    "file_name": image_file_name,
                    "height": data["imageHeight"],
                    "width": data["imageWidth"],
                    "id": cur_image_idx
                })
                cur_image_idx += 1

            image_id = images_with_idx[image_file_name]
            annotations += labelme_polygons_data_to_coco_annotation(data, image_id, cur_polygon_idx)
            cur_polygon_idx += len(data["shapes"])
    
    # include annotations json
    output_coco_json_data["annotations"] = annotations

    # include images jsons
    output_coco_json_data["images"] = coco_images

    with open(output_coco_json_file, "w") as f:
        json.dump(output_coco_json_data, f, indent=4)



def labelme_polygons_data_to_coco_annotation(polygons_data, image_id, start_polygon_idx):
    annotations = []
    height = polygons_data["imageHeight"]
    width = polygons_data["imageWidth"]

    polygon_idx = start_polygon_idx
    for shape in polygons_data["shapes"]:
        if shape["shape_type"] != "polygon":
            continue

        label = shape["label"]
        points = shape["points"]

        # Create binary mask from polygon
        mask = utils.shape_to_mask(
            img_shape=(height, width),
            points=points,
            shape_type="polygon"
        )

        # Convert mask to RLE (Fortran order required)
        rle = maskUtils.encode(
            np.asfortranarray(mask.astype(np.uint8))
        )

        # Convert bytes to string for JSON serialization
        rle["counts"] = rle["counts"].decode("utf-8")

        # Compute area
        area = maskUtils.area(rle).item()

        annotations.append({
            "segmentation": rle,
            "area": area,
            "isCrowd": 0,
            "bbox": mask_to_bbox(mask),
            "category_id": int(label) if label.isdigit() else label,
            "image_id": image_id,
            "id": polygon_idx
        })

        polygon_idx += 1

    return annotations

c:\Users\2DFab\Documents\Software\autoflakedet\workspace-rk


In [35]:
path_to_polygons = current_directory + "\\" + "Labeled Samples" + "\\" + "playground" + "\\" + "labelme_annotations"
output_coco_location = current_directory + "\\" + "Labeled Samples" + "\\" + "playground" + "\\" + "coco_annotations" + "\\" + "coco_output.json"
create_bulk_coco_annotations_from_polygon_images(path_to_polygons, output_coco_location)

251117_c1_f2_50x.png c:\Users\2DFab\Documents\Software\autoflakedet\workspace-rk\Graphene-285-nm-samples\251117_c1_f2_50x.png
c3_f1_20x.jpg c:\Users\2DFab\Documents\Software\autoflakedet\workspace-rk\Graphene-285-nm-samples\c3_f1_20x.jpg
